In [1]:
import pandas as pd
import numpy as np
import glob
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score, precision_recall_curve

In [2]:
all_files = glob.glob("../data/PRICE_AND_DEMAND_*_SA1.csv")
print(f"Found {len(all_files)} files")

df = pd.concat([pd.read_csv(f) for f in all_files], ignore_index=True)
df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"])
df = df[df["REGION"] == "SA1"].sort_values("SETTLEMENTDATE").reset_index(drop=True)
print(df.shape)

Found 12 files
(103680, 5)


In [3]:
df["hour"] = df["SETTLEMENTDATE"].dt.hour
df["day_of_week"] = df["SETTLEMENTDATE"].dt.dayofweek
df["day_of_year"] = df["SETTLEMENTDATE"].dt.dayofyear
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["doy_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)

In [4]:
# Same target and lag features as the logistic regression baseline (0.2), plus a
# rolling std the tree model can use to capture how volatile demand currently is.
INTERVALS_PER_HOUR = 12
THRESHOLD = 200
LOOKAHEAD = 3 * INTERVALS_PER_HOUR

df["will_breach"] = (
    df["TOTALDEMAND"].shift(-1).rolling(window=LOOKAHEAD, min_periods=LOOKAHEAD).min() < THRESHOLD
).astype(int)

df["demand_lag_30m"] = df["TOTALDEMAND"].shift(6)
df["demand_lag_1h"] = df["TOTALDEMAND"].shift(12)
df["demand_lag_2h"] = df["TOTALDEMAND"].shift(24)
df["demand_trend"] = df["TOTALDEMAND"] - df["demand_lag_2h"]
df["demand_rolling_std_2h"] = df["TOTALDEMAND"].rolling(24).std()

gap_minutes = df["SETTLEMENTDATE"].diff().dt.total_seconds() / 60
is_gap_nearby = (
    gap_minutes.rolling(LOOKAHEAD, min_periods=1).max().gt(5)
    | gap_minutes.shift(-LOOKAHEAD).rolling(LOOKAHEAD, min_periods=1).max().gt(5)
)
df = df[~is_gap_nearby.fillna(True)]
df = df.dropna().reset_index(drop=True)
print(df.shape)
print(f"Positive class rate: {df['will_breach'].mean():.3%}")

(103368, 20)
Positive class rate: 4.707%


In [5]:
df["year"] = df["SETTLEMENTDATE"].dt.year
train = df[df["year"] <= 2025]
test = df[df["year"] == 2026]
print(f"Train: {len(train)} rows, breach rate {train['will_breach'].mean():.3%}")
print(f"Test: {len(test)} rows, breach rate {test['will_breach'].mean():.3%}")

Train: 77807 rows, breach rate 5.100%
Test: 25561 rows, breach rate 3.513%


In [6]:
features = ["hour_sin", "hour_cos", "doy_sin", "doy_cos", "dow_sin", "dow_cos",
            "demand_lag_30m", "demand_lag_1h", "demand_lag_2h", "demand_trend",
            "demand_rolling_std_2h"]

X_train, y_train = train[features], train["will_breach"]
X_test, y_test = test[features], test["will_breach"]

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print(f"Average precision (PR-AUC): {average_precision_score(y_test, y_proba):.3f}")

scale_pos_weight: 18.61


              precision    recall  f1-score   support

           0       1.00      1.00      1.00     24663
           1       0.90      0.92      0.91       898

    accuracy                           0.99     25561
   macro avg       0.95      0.96      0.95     25561
weighted avg       0.99      0.99      0.99     25561

Average precision (PR-AUC): 0.977


In [7]:
# Compare against the 0.2 logistic regression baseline on the identical train/test split.
baseline_features = ["hour_sin", "hour_cos", "doy_sin", "doy_cos",
                      "demand_lag_30m", "demand_lag_1h", "demand_lag_2h", "demand_trend"]

baseline = LogisticRegression(class_weight="balanced", max_iter=1000)
baseline.fit(train[baseline_features], y_train)
baseline_proba = baseline.predict_proba(test[baseline_features])[:, 1]

print(f"Logistic regression PR-AUC: {average_precision_score(y_test, baseline_proba):.3f}")
print(f"XGBoost PR-AUC:             {average_precision_score(y_test, y_proba):.3f}")

Logistic regression PR-AUC: 0.934
XGBoost PR-AUC:             0.977


In [8]:
importances = pd.Series(model.feature_importances_, index=features).sort_values(ascending=False)
print(importances)

demand_lag_1h            0.388506
demand_lag_2h            0.375980
demand_lag_30m           0.079257
demand_trend             0.040829
demand_rolling_std_2h    0.035778
hour_sin                 0.019151
hour_cos                 0.017873
doy_sin                  0.013397
dow_sin                  0.010876
dow_cos                  0.009696
doy_cos                  0.008656
dtype: float32


## Feature-matched comparison and demand-floor baselines

Compare the full 11-feature XGBoost model against an XGBoost trained on only the 8 baseline features (matching 0.2 exactly), a naive "current demand < 400 MW" persistence rule, and all four models restricted to test rows where demand hasn't already dropped below the 200 MW threshold (i.e. rows where the model still has to *forecast* a breach rather than just observe one).

In [9]:
from sklearn.metrics import precision_score, recall_score

# XGBoost restricted to the same 8 features as the 0.2 logistic regression baseline,
# so any gap vs. the 11-feature model above is attributable to the extra features,
# not to the model class.
model_8feat = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=42,
)
model_8feat.fit(train[baseline_features], y_train)

proba_8feat = model_8feat.predict_proba(test[baseline_features])[:, 1]
pred_8feat = model_8feat.predict(test[baseline_features])

print(f"Logistic regression PR-AUC:        {average_precision_score(y_test, baseline_proba):.3f}")
print(f"XGBoost (8 features, feat-matched) PR-AUC: {average_precision_score(y_test, proba_8feat):.3f}")
print(f"XGBoost (11 features) PR-AUC:      {average_precision_score(y_test, y_proba):.3f}")

Logistic regression PR-AUC:        0.934
XGBoost (8 features, feat-matched) PR-AUC: 0.970
XGBoost (11 features) PR-AUC:      0.977


In [10]:
# Naive persistence baseline: "it's already close to the threshold, so it'll breach."
# No model, no lookahead reasoning -- just a fixed floor on current demand.
PERSISTENCE_THRESHOLD = 400
persistence_pred = (test["TOTALDEMAND"] < PERSISTENCE_THRESHOLD).astype(int).to_numpy()

print(f"Persistence rule flags {persistence_pred.mean():.3%} of test rows as breaches")
print(classification_report(y_test, persistence_pred))

Persistence rule flags 4.350% of test rows as breaches
              precision    recall  f1-score   support

           0       0.99      0.98      0.98     24663
           1       0.56      0.70      0.62       898

    accuracy                           0.97     25561
   macro avg       0.78      0.84      0.80     25561
weighted avg       0.97      0.97      0.97     25561



In [11]:
# Restrict to rows where current demand hasn't already crossed the 200 MW threshold --
# this is the regime where a forecast actually adds value over just reading the current value.
subset_mask = (test["TOTALDEMAND"] >= 200).to_numpy()
y_test_arr = y_test.to_numpy()
n_subset = subset_mask.sum()
n_positive_subset = int(y_test_arr[subset_mask].sum())
print(f"Test rows with TOTALDEMAND >= 200: {n_subset} ({n_positive_subset} positive breach events)")

baseline_pred = baseline.predict(test[baseline_features])

model_outputs = {
    "Logistic regression (8 feat)": (baseline_proba, baseline_pred),
    "XGBoost (8 feat, matched)": (proba_8feat, pred_8feat),
    "XGBoost (11 feat)": (y_proba, y_pred),
    "Persistence (<400 MW)": (persistence_pred.astype(float), persistence_pred),
}


def eval_scores(y_true, y_score, y_pred, mask=None):
    if mask is not None:
        y_true, y_score, y_pred = y_true[mask], y_score[mask], y_pred[mask]
    return pd.Series({
        "PR-AUC": average_precision_score(y_true, y_score),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
    })


all_test = pd.DataFrame({
    name: eval_scores(y_test_arr, score, pred)
    for name, (score, pred) in model_outputs.items()
}).T

subset_test = pd.DataFrame({
    name: eval_scores(y_test_arr, score, pred, mask=subset_mask)
    for name, (score, pred) in model_outputs.items()
}).T

comparison = pd.concat(
    {"All test rows": all_test, "Test rows, demand >= 200": subset_test}, axis=1
)
comparison.round(3)

Test rows with TOTALDEMAND >= 200: 25158 (495 positive breach events)


All test rows                   \
                                    PR-AUC Precision Recall   
Logistic regression (8 feat)         0.934     0.718  0.937   
XGBoost (8 feat, matched)            0.970     0.887  0.909   
XGBoost (11 feat)                    0.977     0.895  0.915   
Persistence (<400 MW)                0.404     0.564  0.698   

                             Test rows, demand >= 200                   
                                               PR-AUC Precision Recall  
Logistic regression (8 feat)                    0.878     0.581  0.925  
XGBoost (8 feat, matched)                       0.921     0.801  0.846  
XGBoost (11 feat)                               0.938     0.816  0.859  
Persistence (<400 MW)                           0.154     0.316  0.453